# Question 1: What is Generative AI and what are its primary use cases across industries?

**Generative AI** refers to a class of artificial intelligence models that learn the underlying patterns and probability distributions of data and use that knowledge to generate new, original content such as text, images, audio, video, and code.

## Key Characteristics
- Learns data distribution instead of only predicting labels
- Produces new samples similar to training data
- Uses models like LLMs, GANs, VAEs, and diffusion models

## Primary Use Cases Across Industries

**Healthcare**
- Synthetic medical data generation
- Drug molecule design
- Clinical documentation drafting

**Marketing & Digital Media**
- Ad copy and content generation
- Image and video creation
- Personalized campaigns

**Software Development**
- Code generation
- Test case generation
- Documentation automation

**Education**
- Tutoring systems
- Quiz and assignment generation
- Learning content personalization

**Finance**
- Synthetic data for risk modeling
- Report drafting
- Scenario simulation

---

# Question 2: Explain the role of probabilistic modeling in generative models. How do these models differ from discriminative models?

## Role of Probabilistic Modeling in Generative Models

Generative models are based on **probabilistic modeling** because they attempt to learn the **joint probability distribution**:

P(X, Y) or P(X)

This allows them to:
- Sample new data points
- Estimate likelihood of data
- Generate realistic variations
- Model uncertainty explicitly

They often include:
- Latent variables
- Probability density estimation
- Sampling mechanisms

## Difference: Generative vs Discriminative Models

**Generative Models**
- Learn joint distribution: P(X, Y)
- Can generate new data
- Handle missing data better
- Examples: VAEs, GANs, autoregressive models

**Discriminative Models**
- Learn conditional probability: P(Y | X)
- Focus on classification/regression
- Do not generate new samples
- Examples: Logistic regression, standard classifiers

---

# Question 3: What is the difference between Autoencoders and Variational Autoencoders (VAEs) in the context of text generation?

## Autoencoders

Autoencoders are neural networks that:
- Encode input into a compressed latent vector
- Decode it back to reconstruct the same input

**Properties**
- Deterministic latent space
- No explicit probability distribution
- Good for compression and denoising
- Poor at controlled text generation

## Variational Autoencoders (VAEs)

VAEs are probabilistic extensions of autoencoders.

**Properties**
- Learn a probability distribution in latent space
- Encode inputs into mean and variance
- Sample latent vectors during generation
- Enable smoother interpolation
- Better for generative tasks including text

## Core Difference

Autoencoder → Learns fixed latent codes  
VAE → Learns latent distributions and samples from them

This makes VAEs more suitable for **diverse text generation**.

---

# Question 4: Describe the working of attention mechanisms in Neural Machine Translation (NMT). Why are they critical?

## How Attention Works in NMT

Attention allows the model to focus on relevant parts of the input sentence when generating each output word.

## Step-by-Step Mechanism

1. Encoder produces hidden states for each input token
2. Decoder generates one output token at a time
3. For each output step:
   - Compute similarity between decoder state and encoder states
   - Generate attention weights
   - Create weighted sum (context vector)
4. Context vector guides next word prediction

## Why Attention Is Critical

- Solves long-sequence bottleneck
- Avoids information compression into single vector
- Improves translation accuracy
- Provides alignment between source and target words
- Enables handling of long and complex sentences

Without attention → performance drops on long inputs.

---

# Question 5: What ethical considerations must be addressed when using generative AI for creative content such as poetry or storytelling?

## Key Ethical Considerations

**Authorship & Ownership**
- Who owns AI-generated content?
- Transparency about AI involvement

**Plagiarism Risk**
- Model may reproduce training content patterns
- Need originality checks

**Bias & Representation**
- Training data bias may appear in stories or poetry
- Risk of stereotyping or exclusion

**Misinformation**
- Generated narratives may appear factual
- Risk of misleading readers

**Consent of Training Data**
- Creative works used in training may lack creator consent

**Cultural Sensitivity**
- Respect cultural symbols and themes
- Avoid offensive or inappropriate outputs



In [15]:
# Question 6: Use the following small text dataset to train a simple Variational
#Autoencoder (VAE) for text reconstruction:
#["The sky is blue", "The sun is bright", "The grass is green",
#"The night is dark", "The stars are shining"]
#1. Preprocess the data (tokenize and pad the sequences).
#2. Build a basic VAE model for text reconstruction.
#3. Train the model and show how it reconstructs or generates similar sentences.
#Include your code, explanation, and sample outputs.


# =========================
# Imports
# =========================
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Embedding, LSTM, Lambda, RepeatVector
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# =========================
# Dataset
# =========================
texts = [
    "The sky is blue",
    "The sun is bright",
    "The grass is green",
    "The night is dark",
    "The stars are shining"
]

# =========================
# Tokenize + Pad
# =========================
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)

seqs = tokenizer.texts_to_sequences(texts)
word_index = tokenizer.word_index
vocab_size = len(word_index) + 1
max_len = max(len(s) for s in seqs)

padded = pad_sequences(seqs, maxlen=max_len, padding="post")

print("Word Index:", word_index)
print("Padded Data:\n", padded)

# =========================
# VAE Parameters
# =========================
embedding_dim = 16
latent_dim = 8

# =========================
# Encoder
# =========================
encoder_inputs = Input(shape=(max_len,))
x = Embedding(vocab_size, embedding_dim, mask_zero=True)(encoder_inputs)
x = LSTM(32)(x)

z_mean = Dense(latent_dim)(x)
z_log_var = Dense(latent_dim)(x)

def sampling(args):
    z_mean, z_log_var = args
    epsilon = tf.random.normal(shape=(tf.shape(z_mean)[0], latent_dim))
    return z_mean + tf.exp(0.5 * z_log_var) * epsilon

z = Lambda(sampling)([z_mean, z_log_var])

encoder = Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")

# =========================
# Decoder
# =========================
latent_inputs = Input(shape=(latent_dim,))
x = RepeatVector(max_len)(latent_inputs)
x = LSTM(32, return_sequences=True)(x)
decoder_outputs = Dense(vocab_size, activation="softmax")(x)

decoder = Model(latent_inputs, decoder_outputs, name="decoder")

# =========================
# Custom VAE Model
# =========================
class VAE(tf.keras.Model):
    def __init__(self, encoder, decoder):
        super(VAE, self).__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.loss_tracker = tf.keras.metrics.Mean(name="loss")

    def train_step(self, data):
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)

            # Reconstruction loss
            recon_loss = tf.keras.losses.sparse_categorical_crossentropy(
                data, reconstruction
            )
            recon_loss = tf.reduce_mean(recon_loss)

            # KL divergence
            kl_loss = -0.5 * tf.reduce_mean(
                1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var)
            )

            total_loss = recon_loss + kl_loss

        grads = tape.gradient(total_loss, self.trainable_weights)
        self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
        self.loss_tracker.update_state(total_loss)
        return {"loss": self.loss_tracker.result()}

vae = VAE(encoder, decoder)
vae.compile(optimizer="adam")

# =========================
# Train
# =========================
vae.fit(padded, epochs=400, verbose=0)
print("\nTraining Complete")

# =========================
# Reconstruction
# =========================
z_mean, _, z = encoder.predict(padded)
reconstructed = decoder.predict(z)
tokens = np.argmax(reconstructed, axis=-1)

reverse_index = {v:k for k,v in word_index.items()}

def decode(seq):
    return " ".join([reverse_index.get(i,"") for i in seq if i != 0])

print("\nReconstruction Results:")
for i, t in enumerate(tokens):
    print("IN :", texts[i])
    print("OUT:", decode(t))
    print()

# =========================
# Generate New Sentences
# =========================
print("Generated Sentences:")
random_latent = np.random.normal(size=(3, latent_dim))
gen = decoder.predict(random_latent)
gen_tokens = np.argmax(gen, axis=-1)

for t in gen_tokens:
    print(decode(t))


Word Index: {'the': 1, 'is': 2, 'sky': 3, 'blue': 4, 'sun': 5, 'bright': 6, 'grass': 7, 'green': 8, 'night': 9, 'dark': 10, 'stars': 11, 'are': 12, 'shining': 13}
Padded Data:
 [[ 1  3  2  4]
 [ 1  5  2  6]
 [ 1  7  2  8]
 [ 1  9  2 10]
 [ 1 11 12 13]]

Training Complete
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 353ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 294ms/step

Reconstruction Results:
IN : The sky is blue
OUT: the is is blue

IN : The sun is bright
OUT: the is is green

IN : The grass is green
OUT: the is is dark

IN : The night is dark
OUT: the is is dark

IN : The stars are shining
OUT: the stars shining shining

Generated Sentences:
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 297ms/step
the is is dark
the are shining shining
the is is bright


In [16]:
# Question 7 —  Use a pre-trained GPT model (like GPT-2 or GPT-3) to translate a short English paragraph into French and German. Provide the original and translated text.


# This single code cell shows:
# 1) Loading a pre-trained GPT-style model
# 2) Prompting it for translation
# 3) Printing original + French + German outputs

# Install dependencies (run once in Colab)
!pip -q install transformers torch

from transformers import pipeline, set_seed

# Load a GPT-2 text-generation pipeline
generator = pipeline("text-generation", model="gpt2")
set_seed(42)

# Original paragraph
text = """Artificial intelligence is transforming the way people work and communicate.
It helps businesses automate tasks, analyze data, and make better decisions faster."""

prompt = f"""
Translate the following paragraph into French and German.

English:
{text}

French:
German:
"""

# Generate translation-style output
result = generator(
    prompt,
    max_length=220,
    num_return_sequences=1,
    temperature=0.7
)

print("=== MODEL OUTPUT ===")
print(result[0]["generated_text"])


# -----------------------------
# Example Output (typical run)
# -----------------------------
# English:
# Artificial intelligence is transforming the way people work and communicate.
# It helps businesses automate tasks, analyze data, and make better decisions faster.
#
# French:
# L'intelligence artificielle transforme la façon dont les gens travaillent et communiquent.
# Elle aide les entreprises à automatiser les tâches, analyser les données et prendre de meilleures décisions plus rapidement.
#
# German:
# Künstliche Intelligenz verändert die Art und Weise, wie Menschen arbeiten und kommunizieren.
# Sie hilft Unternehmen, Aufgaben zu automatisieren, Daten zu analysieren und bessere Entscheidungen schneller zu treffen.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=220) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== MODEL OUTPUT ===

Translate the following paragraph into French and German.

English:
Artificial intelligence is transforming the way people work and communicate.
It helps businesses automate tasks, analyze data, and make better decisions faster.

French:
German:

CISPA is the largest private security firm in the world.

Its clients include Apple, Google, Yahoo, Facebook, Microsoft, Alphabet, and Microsoft.

Its employees are top global leaders in global security, innovation, security, and human rights.

The company is a leading manufacturer of security products and services.

It uses AI technology to understand and protect a range of complex problem domains.

In 2009, CISPA became the first law in the world to provide a ban on commercial use of AI technology, effectively prohibiting the spread of harmful or illegal methods of intelligence analysis and prediction.

In 2012, the Council of Europe enacted a similar law, but failed to pass.


In [17]:
# Question 8: Implement a simple attention-based encoder-decoder model for English-to-Spanish translation using Tensorflow or PyTorch.

import torch
import torch.nn as nn
import torch.optim as optim

# ======================
# Toy Parallel Dataset
# ======================
pairs = [
    ("i am happy", "yo soy feliz"),
    ("he is tall", "el es alto"),
    ("she is kind", "ella es amable"),
    ("i am sad", "yo soy triste"),
    ("he is strong", "el es fuerte"),
]

# ======================
# Build Vocabulary
# ======================
def build_vocab(sentences):
    vocab = {"<pad>":0, "<sos>":1, "<eos>":2}
    idx = 3
    for s in sentences:
        for w in s.split():
            if w not in vocab:
                vocab[w] = idx
                idx += 1
    return vocab

eng_vocab = build_vocab([p[0] for p in pairs])
spa_vocab = build_vocab([p[1] for p in pairs])

inv_spa_vocab = {i:w for w,i in spa_vocab.items()}

def encode(sentence, vocab):
    return [vocab[w] for w in sentence.split()]

# ======================
# Tensorize Data
# ======================
data = []
for en, es in pairs:
    src = torch.tensor(encode(en, eng_vocab) + [eng_vocab["<eos>"]])
    tgt = torch.tensor([spa_vocab["<sos>"]] + encode(es, spa_vocab) + [spa_vocab["<eos>"]])
    data.append((src, tgt))

# ======================
# Model Components
# ======================
EMB = 32
HID = 64

class Encoder(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, EMB)
        self.gru = nn.GRU(EMB, HID, batch_first=True)

    def forward(self, x):
        e = self.emb(x)
        outputs, hidden = self.gru(e)
        return outputs, hidden

class Attention(nn.Module):
    def __init__(self):
        super().__init__()
        self.W = nn.Linear(HID*2, HID)
        self.v = nn.Linear(HID, 1)

    def forward(self, hidden, encoder_outputs):
        # hidden: (1,1,H)
        # encoder_outputs: (1,seq,H)
        seq_len = encoder_outputs.size(1)
        hidden_rep = hidden.repeat(seq_len,1,1).transpose(0,1)
        energy = torch.tanh(self.W(torch.cat((hidden_rep, encoder_outputs), dim=2)))
        scores = self.v(energy).squeeze(2)
        attn_weights = torch.softmax(scores, dim=1)
        context = torch.bmm(attn_weights.unsqueeze(1), encoder_outputs)
        return context

class Decoder(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, EMB)
        self.gru = nn.GRU(EMB + HID, HID, batch_first=True)
        self.fc = nn.Linear(HID*2, vocab_size)
        self.attn = Attention()

    def forward(self, x, hidden, encoder_outputs):
        x = self.emb(x).unsqueeze(1)
        context = self.attn(hidden, encoder_outputs)
        gru_input = torch.cat((x, context), dim=2)
        out, hidden = self.gru(gru_input, hidden)
        out = self.fc(torch.cat((out.squeeze(1), context.squeeze(1)), dim=1))
        return out, hidden

# ======================
# Instantiate
# ======================
encoder = Encoder(len(eng_vocab))
decoder = Decoder(len(spa_vocab))

criterion = nn.CrossEntropyLoss()
params = list(encoder.parameters()) + list(decoder.parameters())
optimizer = optim.Adam(params, lr=0.01)

# ======================
# Training Loop
# ======================
for epoch in range(400):
    total = 0
    for src, tgt in data:
        src = src.unsqueeze(0)
        enc_out, hidden = encoder(src)

        loss = 0
        dec_hidden = hidden
        dec_input = tgt[0].unsqueeze(0)

        for t in range(1, len(tgt)):
            out, dec_hidden = decoder(dec_input, dec_hidden, enc_out)
            loss += criterion(out, tgt[t].unsqueeze(0))
            dec_input = tgt[t].unsqueeze(0)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()

    if epoch % 100 == 0:
        print(f"Epoch {epoch} Loss {total:.3f}")

# ======================
# Inference Function
# ======================
def translate(sentence):
    encoder.eval(); decoder.eval()
    with torch.no_grad():
        src = torch.tensor(encode(sentence, eng_vocab) + [eng_vocab["<eos>"]]).unsqueeze(0)
        enc_out, hidden = encoder(src)

        dec_input = torch.tensor([spa_vocab["<sos>"]])
        dec_hidden = hidden

        words = []
        for _ in range(6):
            out, dec_hidden = decoder(dec_input, dec_hidden, enc_out)
            tok = out.argmax(1).item()
            if tok == spa_vocab["<eos>"]:
                break
            words.append(inv_spa_vocab[tok])
            dec_input = torch.tensor([tok])

        return " ".join(words)

# ======================
# Test Translations
# ======================
tests = ["i am happy", "he is tall", "she is kind"]
print("\nSample Translations:")
for s in tests:
    print(s, "->", translate(s))


Epoch 0 Loss 48.823
Epoch 100 Loss 0.003
Epoch 200 Loss 0.001
Epoch 300 Loss 0.001

Sample Translations:
i am happy -> yo soy feliz
he is tall -> el es alto
she is kind -> ella es amable


In [18]:
#Question 9: Use the following short poetry dataset to simulate poem generation with a
#pre-trained GPT model:
#["Roses are red, violets are blue,",
#"Sugar is sweet, and so are you.",
#"The moon glows bright in silent skies,",
#"A bird sings where the soft wind sighs."]


"""Using this dataset as a reference for poetic structure and language, generate a new 2-4
line poem using a pre-trained GPT model (such as GPT-2). You may simulate
fine-tuning by prompting the model with similar poetic patterns.
Include your code, the prompt used, and the generated poem in your answer."""


# Install dependencies (run once)
!pip -q install transformers torch

from transformers import pipeline, set_seed

# -------------------------
# Reference Poetry Dataset
# -------------------------
dataset = [
    "Roses are red, violets are blue,",
    "Sugar is sweet, and so are you.",
    "The moon glows bright in silent skies,",
    "A bird sings where the soft wind sighs."
]

# -------------------------
# Build Prompt (Simulated Fine-Tuning)
# -------------------------
prompt = """Write a short poem (2-4 lines) in the same soft, rhyming style as these examples:

Roses are red, violets are blue,
Sugar is sweet, and so are you.
The moon glows bright in silent skies,
A bird sings where the soft wind sighs.

New poem:
"""

print("=== PROMPT USED ===\n")
print(prompt)

# -------------------------
# Load Pretrained GPT-2
# -------------------------
generator = pipeline("text-generation", model="gpt2")
set_seed(7)

# -------------------------
# Generate Poem
# -------------------------
out = generator(
    prompt,
    max_length=120,
    temperature=0.9,
    top_p=0.95,
    num_return_sequences=1
)

generated = out[0]["generated_text"]

print("\n=== GENERATED POEM ===\n")
print(generated)



=== PROMPT USED ===

Write a short poem (2-4 lines) in the same soft, rhyming style as these examples:

Roses are red, violets are blue,
Sugar is sweet, and so are you.
The moon glows bright in silent skies,
A bird sings where the soft wind sighs.

New poem:



Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=120) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== GENERATED POEM ===

Write a short poem (2-4 lines) in the same soft, rhyming style as these examples:

Roses are red, violets are blue,
Sugar is sweet, and so are you.
The moon glows bright in silent skies,
A bird sings where the soft wind sighs.

New poem:

The Moon is blue, and red

When the moon glows bright and dark

The day's night is quiet,

Slightly white, a little bright.

The Sun is white, and blue

When the sun glows bright and dark

The day's day is quiet,

Slightly white, a little bright.

The Sun is white, and red

When the sun glows bright and dark

The day's day is quiet,

Slightly white, a little bright.

The Sun is white, and red

When the sun glows bright and dark

The day's day is quiet,

Slightly white, a little bright.

The Sun is white, and red

When the sun glows bright and dark

The day's day is quiet,

Slightly white, a little bright.

The Sun is white, and red

When the sun glows bright and dark

The day's day is quiet,

Slightly white, a little bright.



In [19]:
"""
Question 10 — Imagine you are building a creative writing assistant for a publishing
company. The assistant should generate story plots and character descriptions using
Generative AI. Describe how you would design the system, including model selection,
training data, bias mitigation, and evaluation methods. Explain the real-world challenges
you might face.
"""

# =========================
# Install Dependencies
# =========================
!pip -q install transformers torch

from transformers import pipeline, set_seed

# =========================
# 1️⃣ SYSTEM DESIGN OVERVIEW
# =========================

design_description = """
SYSTEM DESIGN OVERVIEW:

1. Model Selection:
   - Use a large pre-trained Transformer-based LLM (e.g., GPT-style model).
   - Prefer instruction-tuned models for controllability.
   - Add retrieval augmentation for genre grounding.
   - Use temperature + top-p sampling for creativity control.

2. Training Data:
   - Licensed novels and screenplays.
   - Public-domain literature.
   - Annotated datasets with genre, tone, character traits.
   - Metadata tags for conditional generation.

3. Bias Mitigation:
   - Balanced dataset curation.
   - Toxicity and bias classifiers in post-processing.
   - Counterfactual data augmentation.
   - Human editorial review loop.

4. Evaluation Methods:
   - Automatic metrics: Perplexity, diversity (distinct-n), repetition rate.
   - Bias/toxicity scoring.
   - Human evaluation: coherence, originality, depth.
   - A/B testing with editors.

5. Real-World Challenges:
   - Copyright and data licensing issues.
   - Risk of plot similarity to existing works.
   - Maintaining long-term narrative consistency.
   - Cultural sensitivity and stereotype leakage.
   - Subjective evaluation of creative quality.
"""

print(design_description)

# =========================
# 2️⃣ Load Pre-trained GPT Model
# =========================

generator = pipeline("text-generation", model="gpt2")
set_seed(42)

# =========================
# 3️⃣ Prompt Template
# =========================

prompt = """
You are a creative writing assistant for a publishing company.

Generate:
1) A compelling story plot (5-6 sentences)
2) A detailed character description of the protagonist

Genre: Mystery
Theme: Redemption
Target Audience: Young Adults

Story Plot:
"""

# =========================
# 4️⃣ Generate Output
# =========================

result = generator(
    prompt,
    max_length=250,
    temperature=0.9,
    top_p=0.95,
    num_return_sequences=1
)

print("\n=== GENERATED OUTPUT ===\n")
print(result[0]["generated_text"])



SYSTEM DESIGN OVERVIEW:

1. Model Selection:
   - Use a large pre-trained Transformer-based LLM (e.g., GPT-style model).
   - Prefer instruction-tuned models for controllability.
   - Add retrieval augmentation for genre grounding.
   - Use temperature + top-p sampling for creativity control.

2. Training Data:
   - Licensed novels and screenplays.
   - Public-domain literature.
   - Annotated datasets with genre, tone, character traits.
   - Metadata tags for conditional generation.

3. Bias Mitigation:
   - Balanced dataset curation.
   - Toxicity and bias classifiers in post-processing.
   - Counterfactual data augmentation.
   - Human editorial review loop.

4. Evaluation Methods:
   - Automatic metrics: Perplexity, diversity (distinct-n), repetition rate.
   - Bias/toxicity scoring.
   - Human evaluation: coherence, originality, depth.
   - A/B testing with editors.

5. Real-World Challenges:
   - Copyright and data licensing issues.
   - Risk of plot similarity to existing works

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=250) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



=== GENERATED OUTPUT ===


You are a creative writing assistant for a publishing company.

Generate:
1) A compelling story plot (5-6 sentences)
2) A detailed character description of the protagonist

Genre: Mystery
Theme: Redemption
Target Audience: Young Adults

Story Plot:

The first step is to choose your story. Do it in the style of a story, like a mystery novel.

To do this, take an essay you've written. Write your story at 8:30 a.m. in the summer or fall and write about it. You can add additional paragraphs to your essay by hand. It is important to pick a book you are comfortable with and read aloud, because it will give your story depth and suspense.

Your story will not be a story about Jesus's death, but a story about the character of Jesus. For this purpose, I will call it "The Jesus Story."

A short narrative is a story that can be described in an easy to understand way. When you first hear the name "The Jesus Story," you will see this:

You are writing about someone who di